# Exploration du processed OpenAgenda

Contrôler le résultat du preprocessing avant la préparation pour le RAG : unicité des événements, informations disponibles, catégories et séances. Les données sont uniquement lues ; aucune correction ni sauvegarde de dataset n'est réalisée.

## 1. Chargement et provenance

In [1]:
# ---------- Chargement des données ----------
import json
from html import escape
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

current_directory = Path.cwd().resolve()
project_root = next(
    (directory for directory in (current_directory, *current_directory.parents)
     if (directory / "App" / "Data" / "processed").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Dossier App/Data/processed introuvable depuis ce répertoire.")
processed_directory = project_root / "App" / "Data" / "processed"
files = [path for path in processed_directory.glob("openagenda_processed_*.json") if path.is_file()]
if not files:
    raise FileNotFoundError(f"Aucun fichier openagenda_processed_*.json dans {processed_directory}.")
input_path = max(files, key=lambda path: (path.stat().st_mtime_ns, path.name))
with input_path.open(encoding="utf-8") as source:
    payload = json.load(source)
if not isinstance(payload, dict):
    raise ValueError("Le processed doit être un objet JSON.")
events = payload.get("events")
if not isinstance(events, list) or any(not isinstance(event, dict) for event in events):
    raise ValueError("events doit être une liste d'objets JSON.")

print(f"Fichier processed : {input_path.name}")
print(f"RAW source : {payload.get('source_file')}")
print(f"Date du preprocessing : {payload.get('processed_at')}")
print(f"Événements RAW : {payload.get('raw_event_count')}")
print(f"Doublons supprimés : {payload.get('duplicate_count')}")
print(f"UID invalides ignorés : {payload.get('ignored_uid_count')}")
print(f"Événements processed : {len(events)}")
if payload.get("event_count") != len(events):
    print("Attention : event_count ne correspond pas à la longueur de events.")

Fichier processed : openagenda_processed_20261009T062655426839Z.json
RAW source : openagenda_20261007T123746834561Z.json
Date du preprocessing : 2026-10-09T06:26:55.426839+00:00
Événements RAW : 7222
Doublons supprimés : 3307
UID invalides ignorés : 0
Événements processed : 3915


## 2. Structure et unicité

Vérifier que les champs attendus sont présents et que chaque UID identifie un seul événement. Les anomalies sont signalées, sans correction.

In [2]:
# ---------- Structure du dataset ----------
df = pd.DataFrame(events)
print(f"Lignes : {len(df)} — Colonnes : {len(df.columns)}")
print("Colonnes :", df.columns.tolist())
display(df.head())

if "uid" in df:
    valid_uid = df["uid"].map(
        lambda value: type(value) in (int, str)
        and str(value).isdecimal() and int(value) > 0
    )
    # Normalisation temporaire pour comparer 123 et "123" sans toucher au DataFrame.
    uid_keys = df.loc[valid_uid, "uid"].map(int)
    print(f"UID uniques : {uid_keys.nunique()}")
    print(f"UID dupliqués (occurrences supplémentaires) : {uid_keys.duplicated().sum()}")
    print(f"UID absents ou invalides : {(~valid_uid).sum()}")
else:
    print("Attention : colonne uid absente.")

Lignes : 3915 — Colonnes : 32
Colonnes : ['uid', 'title', 'description', 'long_description', 'keywords', 'timings', 'date_range', 'location_name', 'address', 'postal_code', 'city', 'district', 'metropole', 'department', 'region', 'latitude', 'longitude', 'access', 'event_types', 'public', 'age_min', 'age_max', 'accessibility', 'conditions', 'billetterie', 'event_links', 'online_access_link', 'location_website', 'slug', 'state', 'status', 'origin_agenda']


,uid,title,description,long_description,keywords,timings,date_range,location_name,address,postal_code,...,accessibility,conditions,billetterie,event_links,online_access_link,location_website,slug,state,status,origin_agenda
0,73281314,"Petits mondes lumineux, Cie Les Bricoleuses",Spectacle sous forme d'installation artistique...,### **Petits mondes lumineux est une installat...,"[vacance, vacances d'automne, accessibilité, c...","[{'end': '2026-10-27T11:15:00+01:00', 'id': 'W...","Mardi 27 octobre, 10h30",Muséum de Toulouse,"35 allées Jules Guesde, 31000 Toulouse",31400,...,"[Handicap intellectuel, Handicap psychique]","Inclus dans le billet d'entrée, inscription un...",,[],None,https://museum.toulouse-metropole.fr/,petits-mondes-lumineux-cie-les-bricoleuses,2,Programmé,Muséum de Toulouse
1,359882,Ciné concert C.A.V.E.R.N.E.S.,Plongez dans l'histoire des premiers préhistor...,"### **Le temps d'un ciné concert exceptionnel,...","[vacance, vacances d'automne, expo minéraux, s...","[{'end': '2026-10-28T15:40:00+01:00', 'id': 'D...","Mercredi 28 octobre, 15h00",Muséum de Toulouse,"35 allées Jules Guesde, 31000 Toulouse",31400,...,[],Inclus dans le billet d'entrée,,[],None,https://museum.toulouse-metropole.fr/,cine-concert-cavernes,2,Programmé,Muséum de Toulouse
2,47196421,"Le conteur des cimes : spectacle ""Même pas peur""",À travers ces contes inspirés des traditions p...,"Au cœur des Pyrénées, les grottes de Gargas, B...","[vacance, vacances d'automne, accessibilité, c...","[{'end': '2026-10-29T11:45:00+01:00', 'id': '1...","Jeudi 29 octobre, 11h00, 15h00",Muséum de Toulouse,"35 allées Jules Guesde, 31000 Toulouse",31400,...,"[Handicap intellectuel, Handicap visuel, Handi...",Inclus dans le billet d'entrée,,[],None,https://museum.toulouse-metropole.fr/,le-conteur-des-cimes-spectacle-meme-pas-peur,2,Programmé,Muséum de Toulouse
3,33128226,"Minéraux, bien plus que des cailloux","Entre émerveillement, voyage dans le temps et ...",Exposition temporaire au Muséum de Toulouse\n-...,"[expositions, caillou, mineral, exposition]","[{'end': '2026-10-02T18:00:00+02:00', 'id': 'W...",2 octobre 2026 - 4 juillet 2027,Muséum de Toulouse,"35 allées Jules Guesde, 31000 Toulouse",31400,...,[Handicap moteur],plein tarif 12€ / réduit 8€,None,[{'link': 'https://museum.toulouse-metropole.f...,None,https://museum.toulouse-metropole.fr/,mineraux-bien-plus-que-des-cailloux,2,Programmé,Muséum de Toulouse
4,27548225,"Eugène Trutat - ""Reliefs du temps""",Exposition photographique au Jardin botanique ...,"Scientifique et photographe méthodique, Eugène...","[expositions, expo minéraux]","[{'end': '2026-10-02T18:00:00+02:00', 'id': 'Y...",2 octobre 2026 - 31 janvier 2027,Muséum de Toulouse,"35 allées Jules Guesde, 31000 Toulouse",31400,...,[],Inclus dans le billet d'entrée,None,[{'link': 'https://museum.toulouse-metropole.f...,None,https://museum.toulouse-metropole.fr/,eugene-trutat-reliefs-du-temps,2,Programmé,Muséum de Toulouse


UID uniques : 3915
UID dupliqués (occurrences supplémentaires) : 0
UID absents ou invalides : 0


## 3. Valeurs manquantes

Mesurer la disponibilité de toutes les colonnes, y compris les listes et dictionnaires vides. Un champ absent n'entraîne aucune suppression de ligne.

In [3]:
# ---------- Valeurs manquantes ----------
def is_missing(value):
    """Reconnaître aussi les structures vides, sans appliquer isna à une liste."""
    if value is None:
        return True
    if isinstance(value, str):
        return not value.strip()
    if isinstance(value, (list, dict)):
        return len(value) == 0
    return bool(pd.isna(value))

missing_counts = df.apply(lambda column: column.map(is_missing).sum())
missing_analysis = pd.DataFrame({
    "column": df.columns,
    "present": len(df) - missing_counts.values,
    "missing": missing_counts.values,
    "missing_percent": (
        missing_counts.values / len(df) * 100 if len(df) else missing_counts.values * 0
    ),
}).sort_values("missing_percent", ascending=False).reset_index(drop=True)
missing_analysis["missing_percent"] = missing_analysis["missing_percent"].round(2)
with pd.option_context("display.max_rows", None):
    display(missing_analysis)
if df.empty:
    print("Dataset vide : les pourcentages sont affichés à 0 par convention.")

,column,present,missing,missing_percent
0,online_access_link,51,3864,98.70
1,event_links,967,2948,75.30
2,accessibility,1092,2823,72.11
3,age_min,1595,2320,59.26
4,age_max,1595,2320,59.26
5,billetterie,1627,2288,58.44
6,access,2120,1795,45.85
7,keywords,2274,1641,41.92
8,location_website,2542,1373,35.07
9,conditions,2919,996,25.44


## 4. Types d'événements

Contrôler le décodage des catégories avant d'en mesurer la fréquence. Un événement peut appartenir à plusieurs catégories : la somme peut dépasser le nombre d'événements.

In [4]:
# ---------- Types d'événements ----------
if "event_types" not in df:
    print("Attention : colonne event_types absente.")
else:
    values = df["event_types"]
    invalid_lists = values.map(lambda value: not is_missing(value) and not isinstance(value, list))
    # explode crée une série d'analyse ; la colonne d'origine reste intacte.
    exploded_types = values.explode()
    valid_labels = exploded_types.map(
        lambda value: isinstance(value, str) and bool(value.strip())
        and not value.strip().isdecimal()
    )
    invalid_labels = exploded_types.map(lambda value: not is_missing(value)) & ~valid_labels
    print(f"Valeurs présentes hors liste : {invalid_lists.sum()}")
    print(f"Valeurs non décodées ou invalides : {invalid_labels.sum()}")
    if invalid_labels.any():
        display(exploded_types[invalid_labels].head(10).to_frame("valeur_non_décodée"))

    # Compter une catégorie au plus une fois par événement, sans dédoublonner le dataset.
    type_pairs = pd.DataFrame({
        "event_position": exploded_types.index,
        "event_type": exploded_types.values,
    })
    type_counts = (
        type_pairs.loc[valid_labels.to_numpy()]
        .drop_duplicates(["event_position", "event_type"])["event_type"]
        .value_counts()
        .rename_axis("event_type")
        .reset_index(name="event_count")
    )
    display(type_counts)

    # Graphique SVG : aucune bibliothèque graphique supplémentaire nécessaire.
    if not type_counts.empty:
        width, left, bar_width, row_height = 850, 270, 480, 30
        height = 100 + row_height * len(type_counts)
        maximum = int(type_counts["event_count"].max())
        parts = [
            f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" '
            'role="img" aria-label="Répartition des types d’événements" style="max-width:100%">',
            '<rect width="100%" height="100%" fill="white"/>',
            '<g font-family="sans-serif" font-size="13" fill="#222">',
            '<text x="20" y="24" font-size="17">Répartition des types d’événements</text>',
            '<text x="20" y="46">Type d’événement</text>',
        ]
        for position, row in enumerate(type_counts.itertuples(index=False)):
            y = 60 + position * row_height
            length = row.event_count / maximum * bar_width
            parts.extend([
                f'<text x="{left - 10}" y="{y + 16}" text-anchor="end">{escape(str(row.event_type))}</text>',
                f'<rect x="{left}" y="{y}" width="{length}" height="20" fill="#4477aa"/>',
                f'<text x="{left + length + 6}" y="{y + 16}">{row.event_count}</text>',
            ])
        axis_y = 60 + len(type_counts) * row_height
        parts.extend([
            f'<line x1="{left}" y1="{axis_y}" x2="{left + bar_width}" y2="{axis_y}" stroke="#222"/>',
            f'<text x="{left}" y="{axis_y + 17}">0</text>',
            f'<text x="{left + bar_width}" y="{axis_y + 17}" text-anchor="end">{maximum}</text>',
            f'<text x="{left + bar_width / 2}" y="{axis_y + 35}" text-anchor="middle">Nombre d’événements</text>',
            '</g></svg>',
        ])
        display(HTML("".join(parts)))
    else:
        print("Aucun libellé exploitable pour le graphique.")

Valeurs présentes hors liste : 0
Valeurs non décodées ou invalides : 0


,event_type,event_count
0,Spectacle - Musique,1396
1,Visite,864
2,Stage - Atelier,706
3,Exposition,388
4,Conférence,259
5,Fête - Festival,259
6,Cinéma - Projection,237
7,Événement sportif,202
8,Tous,164
9,Foire - Marchés,28


## 5. Publics

Observer les publics représentés et signaler les valeurs qui n'ont pas été décodées en libellés.

In [5]:
# ---------- Public ----------
if "public" not in df:
    print("Attention : colonne public absente.")
else:
    public_values = df["public"]
    print("Valeurs présentes hors liste :", public_values.map(
        lambda value: not is_missing(value) and not isinstance(value, list)
    ).sum())
    exploded_public = public_values.explode()
    public_labels = exploded_public.map(
        lambda value: isinstance(value, str) and bool(value.strip())
        and not value.strip().isdecimal()
    )
    unresolved_public = exploded_public.map(lambda value: not is_missing(value)) & ~public_labels
    print(f"Valeurs non décodées ou invalides : {unresolved_public.sum()}")
    if unresolved_public.any():
        display(exploded_public[unresolved_public].head(10).to_frame("valeur_non_décodée"))
    public_pairs = pd.DataFrame({
        "event_position": exploded_public.index, "public": exploded_public.values,
    })
    public_counts = (
        public_pairs.loc[public_labels.to_numpy()]
        .drop_duplicates(["event_position", "public"])["public"]
        .value_counts().rename_axis("public").reset_index(name="event_count")
    )
    display(public_counts)

Valeurs présentes hors liste : 0
Valeurs non décodées ou invalides : 0


,public,event_count
0,Tout Public,3380
1,Adulte,940
2,Jeune Public,858
3,En famille,686
4,Étudiant,660
5,Senior,551
6,Personne en situation de handicap,520
7,En groupe,42
8,Professionnel,17


## 6. Dates et séances

Conserver toutes les séances dans les données d'origine. Les dates sont converties temporairement en UTC pour comparer correctement les fuseaux horaires. Les bornes portent sur toutes les séances conservées : un événement sélectionné peut aussi avoir des séances antérieures à la période de collecte.

In [6]:
# ---------- Dates et séances ----------
if "timings" not in df:
    print("Attention : colonne timings absente.")
else:
    timing_values = df["timings"]
    malformed_timings = timing_values.map(
        lambda value: not is_missing(value) and not isinstance(value, list)
    )
    session_counts = timing_values.map(lambda value: len(value) if isinstance(value, list) else 0)
    print(f"Événements sans séance : {(session_counts == 0).sum()}")
    print(f"Événements avec au moins une séance : {(session_counts > 0).sum()}")
    print(f"Champs timings présents hors liste : {malformed_timings.sum()}")
    display(session_counts.rename("session_count").to_frame())
    print(f"Séances par événement — minimum : {session_counts.min()}, "
          f"maximum : {session_counts.max()}, moyenne : {session_counts.mean():.2f}")

    session_rows = []
    invalid_sessions = 0
    for event_position, timings in enumerate(timing_values):
        if isinstance(timings, list):
            for timing in timings:
                if isinstance(timing, dict):
                    session_rows.append({"event_position": event_position, "begin": timing.get("begin")})
                else:
                    invalid_sessions += 1
    sessions = pd.DataFrame(session_rows, columns=["event_position", "begin"])
    sessions["begin_utc"] = pd.to_datetime(
        sessions["begin"], utc=True, errors="coerce", format="mixed"
    )
    print(f"Entrées de séance non structurées : {invalid_sessions}")
    print(f"Dates begin absentes ou invalides : {sessions['begin_utc'].isna().sum()}")
    print(f"Date de séance la plus ancienne (UTC) : {sessions['begin_utc'].min()}")
    print(f"Date de séance la plus récente (UTC) : {sessions['begin_utc'].max()}")
    now = pd.Timestamp.now(tz="UTC")
    past_count = sessions.loc[sessions["begin_utc"] < now, "event_position"].nunique()
    future_count = sessions.loc[sessions["begin_utc"] >= now, "event_position"].nunique()
    print(f"Événements avec au moins une séance passée : {past_count}")
    print(f"Événements avec au moins une séance future : {future_count}")
    print("Un même événement peut figurer dans les deux groupes.")

Événements sans séance : 0
Événements avec au moins une séance : 3915
Champs timings présents hors liste : 0


,session_count
0,1
1,1
2,2
3,234
4,103
...,...
3910,1
3911,1
3912,2
3913,1


Séances par événement — minimum : 1, maximum : 468, moyenne : 6.01
Entrées de séance non structurées : 0
Dates begin absentes ou invalides : 0
Date de séance la plus ancienne (UTC) : 2023-12-13 15:00:00+00:00
Date de séance la plus récente (UTC) : 2028-05-13 18:00:00+00:00
Événements avec au moins une séance passée : 2909
Événements avec au moins une séance future : 1131
Un même événement peut figurer dans les deux groupes.


## 7. Disponibilité pour le RAG et exemple complet

Réutiliser le bilan des valeurs manquantes pour vérifier les informations nécessaires aux questions sur les dates, les expositions, les enfants, l'accessibilité ou les quartiers. La présence d'un champ ne garantit pas à elle seule la qualité de son contenu.

In [7]:
# ---------- Contrôle pour le RAG ----------
rag_fields = [
    "title", "description", "long_description", "keywords", "timings", "date_range",
    "location_name", "address", "city", "district", "event_types", "public",
    "conditions", "accessibility",
]
availability = missing_analysis.set_index("column")["present"].reindex(rag_fields)
absent_columns = availability[availability.isna()].index.tolist()
rag_summary = pd.DataFrame({
    "column": rag_fields,
    "present": availability.to_numpy(),
    "available_percent": (availability.to_numpy() / len(df) * 100
                          if len(df) else availability.to_numpy() * 0),
})
rag_summary["available_percent"] = rag_summary["available_percent"].round(2)
display(rag_summary)
if absent_columns:
    print("Colonnes absentes du dataset :", absent_columns)

# ---------- Exemple complet ----------
if events:
    print("Premier événement processed, avec toutes ses informations :")
    print(json.dumps(events[0], ensure_ascii=False, indent=2))
else:
    print("Aucun événement disponible pour l'exemple.")

,column,present,available_percent
0,title,3915,100.00
1,description,3915,100.00
2,long_description,3882,99.16
3,keywords,2274,58.08
4,timings,3915,100.00
5,date_range,3915,100.00
6,location_name,3915,100.00
7,address,3915,100.00
8,city,3915,100.00
9,district,3907,99.80


Premier événement processed, avec toutes ses informations :
{
  "uid": 73281314,
  "title": "Petits mondes lumineux, Cie Les Bricoleuses",
  "description": "Spectacle sous forme d'installation artistique pour les tout-petits au Muséum.",
  "long_description": "### **Petits mondes lumineux est une installation pour 102 livres, 80 cailloux, 3 artistes, plein d’enfants et tous ceux qui les accompagnent.**\n\nUn opus intime qui cherche à éclairer les premières découvertes des tout-petits, la poésie de leurs premiers pas.  \nC’est une invitation à la lecture et à la rêverie, une proposition atypique où différentes formes s’alternent et se superposent : spectacle, lectures, impromptus et installation dans laquelle le public peut agir. Trois artistes accompagnent pianissimo cette exploration. Elles fabriquent des chemins sonores, tactiles et lumineux pour amener le public à découvrir les livres présents dans l’installation, sur le toit d’une cabane, au creux d’un cailloux, dans les replis d’u